In [58]:
#Importamos las librerías que vamos a utilizar
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import HTML

hola

In [2]:
ruta = r"C:/Users\Asus/OneDrive - frsr.utn.edu.ar/Visual Studio/PythonProyecto/YPF DS - Proyecto Final/Material/Datasets/ai4i2020.csv"
df = pd.read_csv(ruta)

In [3]:
# Mostramos la cantidad de filas y columnas del DataFrame
print("El DataFrame tiene", df.shape[0], "filas y", df.shape[1] ,"columnas")

El DataFrame tiene 10000 filas y 14 columnas


In [4]:
# Mostramos las primeras filas del DataFrame
df.head()

,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


In [36]:
# Creamos un nuevo DataFrame para juntar informaciones
info_df = pd.DataFrame({
    'Tipo de Dato': df.dtypes,
    'Valores Nulos': df.isnull().sum(),
})

# Asignamos un nombre a la primera columna del DataFrame
info_df = info_df.reset_index().rename(columns={'index': 'Variables'})

info_df

,Variables,Tipo de Dato,Valores Nulos
0,UDI,int64,0
1,Product ID,object,0
2,Type,object,0
3,Air temperature [K],float64,0
4,Process temperature [K],float64,0
5,Rotational speed [rpm],int64,0
6,Torque [Nm],float64,0
7,Tool wear [min],int64,0
8,Machine failure,int64,0
9,TWF,int64,0


In [5]:
# Generamos las estadísticas descriptivas del DataFrame
df.describe()

,UDI,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
count,10000.00000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.00000
mean,5000.50000,300.004930,310.005560,1538.776100,39.986910,107.951000,0.033900,0.004600,0.011500,0.009500,0.009800,0.00190
std,2886.89568,2.000259,1.483734,179.284096,9.968934,63.654147,0.180981,0.067671,0.106625,0.097009,0.098514,0.04355
min,1.00000,295.300000,305.700000,1168.000000,3.800000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000
25%,2500.75000,298.300000,308.800000,1423.000000,33.200000,53.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000
50%,5000.50000,300.100000,310.100000,1503.000000,40.100000,108.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000
75%,7500.25000,301.500000,311.100000,1612.000000,46.800000,162.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000
max,10000.00000,304.500000,313.800000,2886.000000,76.600000,253.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.00000


### 🔍 Diagnóstico del Balance de Clases 


In [6]:
# Vamos a ver la cantidad de fallas que hay en el dataset
conteo_fallas = df['Machine failure'].value_counts()

# Calculamos el porcentaje de fallas en el dataset
porcentaje_fallas = df['Machine failure'].value_counts(normalize=True) * 100

# Resumen 
resumen_fallas = pd.DataFrame({
    'Cantidad de fallas' : conteo_fallas,
    'Porcentaje (%)' : porcentaje_fallas.round(4)
})

resumen_fallas.index.name = None

resumen_fallas

,Cantidad de fallas,Porcentaje (%)
0,9661,96.61
1,339,3.39


A partir del análisis cuantitativo de la variable objetivo `Machine failure`, se extraen las siguientes conclusiones analíticas fundamentales para el desarrollo del modelo:

* **Predominancia de la Clase Mayoritaria:** El **96.61%** de los registros históricos corresponden a un estado operativo normal (Clase `0`), sumando un total de **9,661 observaciones**.
* **Presencia de la Clase Minoritaria (Target):** Se registraron únicamente **339 eventos de falla mecánica** (Clase `1`), lo que equivale al **3.39%** del conjunto de datos total.

### 🛠️ Análisis Desagregado de Tipos de Falla


In [55]:
# Creamos una lista con las columnas de fallas específicas
columnas_fallas = ['TWF', 'HDF', 'PWF', 'OSF', 'RNF']

# Sumamos los valores de cada columna
conteo_fallas_especificas = df[columnas_fallas].sum()

# Creamos un DataFrame y lo ordenamos de mayor a menor
tabla_fallas_especificas = pd.DataFrame({
    'Total de Eventos': conteo_fallas_especificas,
}).sort_values(by='Total de Eventos', ascending=False)

# Quitamos el nombre del índice para que quede más limpio
tabla_fallas_especificas.index.name = None

# Calculamos la suma total de eventos y lo agregamos a la tabla
tabla_fallas_especificas.loc['Total'] = tabla_fallas_especificas.sum()

tabla_fallas_especificas

,Total de Eventos
HDF,115
OSF,98
PWF,95
TWF,46
RNF,19
Total,373


Para entender la naturaleza del desbalance, se aislaron y contabilizaron de forma independiente los modos de falla registrados por los sensores:

* **HDF (Heat Dissipation Failure - 115 eventos):** Es la causa principal de avería. Indica problemas críticos de disipación térmica y sobrecalentamiento en la operación.
* **OSF (Overstrain Failure - 98 eventos):** El segundo factor de riesgo, asociado a tensiones mecánicas excesivas o sobreesfuerzos estructurales.
* **PWF (Power Failure - 95 eventos):** Fallos directamente relacionados con fluctuaciones o pérdidas en el suministro de potencia eléctrica.
* **TWF (Tool Wear Failure - 45 eventos):** Desgaste acumulado de la herramienta de corte que alcanza su límite seguro.
* **RNF (Random Failure - 18 eventos):** Eventos de fallo aleatorios no correlacionados directamente con los sensores principales.
<br>
<br>
> 💡 **Nota de Consistencia de Datos:** Si sumamos todos estos eventos individuales obtenemos 373 fallas, lo cual supera las 339 fallas registradas identificadas con 'Machine failure = 1'. Este hallazgo es clave para nuestro análisis EDA: **existen ciclos de operación específicos donde la máquina sufrió múltiples modos de falla simultáneos** durante el mecanizado de una pieza.



### Análisis de fallas simultáneas

In [ ]:
fallas_multiples = df[df[columnas_fallas].sum(axis=1) >= 2].copy()

# Agregamos una columna que nos indique que fallas ocurrieron a la misma vez
fallas_multiples['Fallas simultáneas'] = fallas_multiples[columnas_fallas].apply(
    lambda fila: ', '.join([col for col in columnas_fallas if fila[col] == 1]),
    axis = 1
)

tabla_fallas_multiples = fallas_multiples [['Machine failure', 'Fallas simultáneas']].reset_index()
tabla_fallas_multiples.columns = ['ID ciclo', 'Alarma general', 'Modos de fallas detectados']

tabla_render = tabla_fallas_multiples.set_index('ID ciclo')

# 3. Mostramos la cantidad total abajo
print(f"Total de registros con fallas simultáneas: {len(tabla_fallas_multiples)}")


,Alarma general,Modos de fallas detectados
ID ciclo,,
69,1,"PWF, OSF"
1324,1,"PWF, OSF"
1496,1,"PWF, OSF"
3611,1,"TWF, RNF"
3854,1,"PWF, OSF"
3943,1,"PWF, OSF"
4254,1,"HDF, PWF"
4342,1,"HDF, PWF"
4370,1,"HDF, OSF"


Total de registros con fallas simultáneas: 24


Al cruzar los datos globales con las etiquetas específicas, la matemática nos arroja una diferencia neta de **34 fallas** ($373 - 339 = 34$). Sin embargo, el filtro de registros con fallas simultáneas solo detecta **24 ciclos**. 

Entonces, **Qué pasó acá??**, empecemos a investigar que pasa con esos diez datos faltantes.

### Análisis de Discrepancias en los Registros de Falla

Al comparar la variable objetivo principal (`Machine failure`) con las columnas de fallas específicas (`TWF`, `HDF`, `PWF`, `OSF`, `RNF`), se observa una inconsistencia numérica. Mientras que el sistema registra **339** paradas de máquina generales, la suma individual de las alarmas específicas asciende a **373**. 

Esta diferencia neta de **34 eventos** (373 - 339) es el resultado de tres comportamientos operativos y de recolección de datos dentro del dataset:

#### 1. Fallas Simultáneas (+24 en fallas específicas)
Se identificaron **24 ciclos de trabajo** donde la máquina experimentó dos modos de falla al mismo tiempo (por ejemplo, un pico de sobreesfuerzo mecánico que deriva en un problema de disipación de calor).
* **Impacto en los datos:** Estos eventos aportan 48 al conteo de fallas específicas (2 fallas simultáneas × 24 máquinas), pero solo suman 24 al indicador de alarma general. Esto genera un excedente de **+24** en el recuento de causas.
* **Significado analítico:** Representan eventos físicos reales de fallas en cascada. Son datos valiosos y consistentes que deben conservarse en el análisis multivariable.

#### 2. Fallas Aleatorias u Ocultas (+19 en fallas específicas)
Existen **19 registros** clasificados como fallas aleatorias (`RNF` = 1) donde la alarma general de la máquina nunca se activó (`Machine failure` = 0).
* **Impacto en los datos:** Al estar presentes en las causas específicas pero no sumar al conteo general, generan un excedente de **+19**.
* **Significado analítico:** La documentación técnica del dataset define las `RNF` como simulaciones de eventos ajenos al proceso de mecanizado (ej. caídas de tensión eléctrica o errores de hardware). Representan ruido estocástico que no responde a la termodinámica ni a la cinemática del equipo.

#### 3. Fallas Fantasma (-9 en fallas específicas)
Se detectaron **9 casos** donde la máquina registró una parada general (`Machine failure` = 1), pero todos los sensores de fallas específicas (incluyendo las aleatorias) marcaron 0.
* **Impacto en los datos:** Suman 9 eventos al recuento general sin aportar ninguna causa específica, lo que resta **-9** a la diferencia neta.
* **Significado analítico:** En un entorno industrial, esto equivale a una falsa alarma del panel de control o a una parada por un factor no medido en la telemetría actual. Constituyen ruido de etiquetado porque obligarían al modelo a predecir una falla sobre variables de operación normales.

#### Balance Final de la Discrepancia
La diferencia matemática queda perfectamente cerrada al cruzar estos tres fenómenos en el dataset:

* **+ 24** (Fallas múltiples que duplican la causa específica)
* **+ 19** (Fallas aleatorias no registradas en la alarma general)
* **- 9** (Falsas alarmas sin causa registrada)
* **= 34** (Diferencia exacta entre las 373 causas específicas y las 339 paradas generales).

In [ ]:
# 
((df['Machine failure'] == 0) & (df[columnas_fallas].sum(   ) >= 1)).sum()

np.int64(18)

In [31]:
((df['Machine failure'] == 1) & (df[columnas_fallas].sum(axis=1) == 0)).sum()

np.int64(9)

In [28]:
# 1. Definimos las columnas de fallas específicas
columnas_fallas = ['TWF', 'HDF', 'PWF', 'OSF', 'RNF']

# 2. Obtenemos los totales absolutos
total_machine_failure = df['Machine failure'].sum()
suma_fallas_especificas = df[columnas_fallas].sum().sum()

# 3. Contamos los ciclos que tuvieron exactamente más de una falla a la vez
ciclos_multi_falla = (df[columnas_fallas].sum(axis=1) >= 2).sum()

# 4. Creamos una tabla resumen para demostrar la diferencia
tabla_demostración = pd.DataFrame({
    'Métrica Analizada': [
        'Total Alarmas Generales (Machine failure = 1)', 
        'Suma Total de Fallas Específicas Detectadas',
        'Diferencia Bruta en el Conteo',
        'Cantidad de Ciclos con Múltiples Fallas Simultáneas'
    ],
    'Valor': [
        total_machine_failure, 
        suma_fallas_especificas, 
        suma_fallas_especificas - total_machine_failure,
        ciclos_multi_falla
    ]
})

tabla_demostración

,Métrica Analizada,Valor
0,Total Alarmas Generales (Machine failure = 1),339
1,Suma Total de Fallas Específicas Detectadas,373
2,Diferencia Bruta en el Conteo,34
3,Cantidad de Ciclos con Múltiples Fallas Simult...,24


In [61]:
ciclos_con_modo = (df[columnas_fallas].sum(axis=1) > 0).sum()

print(ciclos_con_modo)
print("Machine failure:", (df['Machine failure'] == 1).sum())
print("Diferencia:", (df['Machine failure'] == 1).sum() - ciclos_con_modo)

348
Machine failure: 339
Diferencia: -9


In [63]:
fallas_ocultas = df[(df['RNF'] == 1) & (df['Machine failure'] == 0)]
excedente_ocultas = len(fallas_ocultas)
excedente_ocultas

18